# Week 3 Assignment: Automated Regional Impact Auditor (ARIA)

## 河川洪災避難所風險評估

**Captain's Log**: Starting the flood risk assessment for Taiwan's shelters. This analysis will combine WRA river data with Fire Agency shelter data to identify high-risk areas and capacity gaps.

**Mission Objectives**:
1. Load and clean river, shelter, and township data
2. Create three-level buffer zones (500m, 1km, 2km) around rivers
3. Assign risk levels to shelters using spatial joins
4. Analyze capacity gaps by administrative districts
5. Create interactive risk map and statistical visualizations

In [ ]:
# Import required libraries
import geopandas as gpd
import pandas as pd
import folium
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os
from urllib.parse import quote
import json
import numpy as np
from IPython.display import display, HTML

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Libraries loaded successfully!")

## 1. Data Loading and Cleaning

**Captain's Log**: Loading the three key datasets - WRA river polygons, Fire Agency shelter data, and TGOS township boundaries. Need to verify CRS and clean coordinate issues.

In [ ]:
# Load environment variables
load_dotenv()

# Get buffer parameters from .env
BUFFER_HIGH = int(os.getenv('BUFFER_HIGH', 500))
BUFFER_MED = int(os.getenv('BUFFER_MED', 1000))
BUFFER_LOW = int(os.getenv('BUFFER_LOW', 2000))

print(f"Buffer parameters: High={BUFFER_HIGH}m, Medium={BUFFER_MED}m, Low={BUFFER_LOW}m")

In [ ]:
# Load WRA river data
print("Loading WRA river data...")
rivers_url = 'https://gic.wra.gov.tw/Gis/gic/API/Google/DownLoad.aspx?fname=RIVERPOLY&filetype=SHP'
rivers = gpd.read_file(rivers_url)

print(f"River data loaded: {len(rivers)} features")
print(f"Original CRS: {rivers.crs}")
print(f"River data columns: {list(rivers.columns)}")
rivers.head()

In [ ]:
# Load shelter data
print("Loading shelter data...")
shelters_csv = pd.read_csv('data/shelters.csv')
print(f"Original shelter data: {len(shelters_csv)} records")

# Check for coordinate issues
print("\nCoordinate quality check:")
print(f"Zero longitude: {(shelters_csv['經度'] == 0).sum()}")
print(f"Zero latitude: {(shelters_csv['緯度'] == 0).sum()}")
print(f"Null longitude: {shelters_csv['經度'].isnull().sum()}")
print(f"Null latitude: {shelters_csv['緯度'].isnull().sum()}")

# Check coordinate ranges (Taiwan should be: lon 119-122, lat 21-26)
valid_lon = (shelters_csv['經度'] >= 119) & (shelters_csv['經度'] <= 122)
valid_lat = (shelters_csv['緯度'] >= 21) & (shelters_csv['緯度'] <= 26)
invalid_coords = ~(valid_lon & valid_lat)
print(f"Invalid coordinate records: {invalid_coords.sum()}")

shelters_csv.head()

In [ ]:
# Clean shelter data
print("Cleaning shelter data...")

# Remove records with invalid coordinates
shelters_clean = shelters_csv[
    (shelters_csv['經度'] != 0) & 
    (shelters_csv['緯度'] != 0) & 
    shelters_csv['經度'].notna() & 
    shelters_csv['緯度'].notna() &
    valid_lon & valid_lat
].copy()

print(f"Before cleaning: {len(shelters_csv)} records")
print(f"After cleaning: {len(shelters_clean)} records")
print(f"Removed: {len(shelters_csv) - len(shelters_clean)} records")

# Convert to GeoDataFrame
shelters = gpd.GeoDataFrame(
    shelters_clean,
    geometry=gpd.points_from_xy(shelters_clean['經度'], shelters_clean['緯度']),
    crs='EPSG:4326'
)

print(f"\nShelter GeoDataFrame created with CRS: {shelters.crs}")
print(f"Shelter columns: {list(shelters.columns)}")
shelters.head()

In [ ]:
# Load township boundaries
print("Loading township boundaries...")
township_url = 'https://www.tgos.tw/tgos/VirtualDir/Product/3fe61d4a-ca23-4f45-8aca-4a536f40f290/' + quote('鄉(鎮、市、區)界線1140318.zip')
townships = gpd.read_file(township_url)

print(f"Township data loaded: {len(townships)} features")
print(f"Township CRS: {townships.crs}")
print(f"Township columns: {list(townships.columns)}")
townships.head()

In [ ]:
# Convert all data to EPSG:3826 (Taiwan's official CRS)
print("Converting all data to EPSG:3826...")

# Rivers (check if already EPSG:3826)
if rivers.crs != 'EPSG:3826':
    rivers = rivers.to_crs('EPSG:3826')
    print(f"Rivers converted to EPSG:3826")
else:
    print("Rivers already in EPSG:3826")

# Shelters
shelters = shelters.to_crs('EPSG:3826')
print(f"Shelters converted to EPSG:3826")

# Townships
townships = townships.to_crs('EPSG:3826')
print(f"Townships converted to EPSG:3826")

print(f"\nFinal CRS check:")
print(f"Rivers: {rivers.crs}")
print(f"Shelters: {shelters.crs}")
print(f"Townships: {townships.crs}")

## 2. Multi-Level Buffer Zone Analysis

**Captain's Log**: Creating three-tier buffer zones around rivers to establish flood risk levels. This is critical for accurate shelter risk assessment.

In [ ]:
# Create three-level buffer zones around rivers
print("Creating multi-level buffer zones...")

# Dissolve all river polygons first for efficiency
rivers_dissolved = rivers.dissolve()
print(f"Dissolved {len(rivers)} river features into {len(rivers_dissolved)} feature")

# Create buffer zones
buffer_high = rivers_dissolved.buffer(BUFFER_HIGH)
buffer_med = rivers_dissolved.buffer(BUFFER_MED)
buffer_low = rivers_dissolved.buffer(BUFFER_LOW)

print(f"Created buffer zones:")
print(f"- High risk (500m): {len(buffer_high)} features")
print(f"- Medium risk (1km): {len(buffer_med)} features")
print(f"- Low risk (2km): {len(buffer_low)} features")

# Convert to GeoDataFrames for spatial operations
buffer_high_gdf = gpd.GeoDataFrame(geometry=buffer_high, crs='EPSG:3826')
buffer_med_gdf = gpd.GeoDataFrame(geometry=buffer_med, crs='EPSG:3826')
buffer_low_gdf = gpd.GeoDataFrame(geometry=buffer_low, crs='EPSG:3826')

print("\nBuffer zones ready for spatial analysis")

## 3. Spatial Join and Risk Level Assignment

**Captain's Log**: Performing spatial joins to identify which shelters fall within each risk buffer zone. Need to handle the one-to-many relationship where shelters may fall in multiple zones.

In [ ]:
# Perform spatial joins for each buffer level
print("Performing spatial joins...")

# High risk shelters
high_risk_join = gpd.sjoin(shelters, buffer_high_gdf, how='left', predicate='within')
high_risk_shelters = high_risk_join[high_risk_join.index_right.notna()].copy()
high_risk_shelters['risk_level'] = 'high'

# Medium risk shelters (not already high risk)
remaining_shelters = shelters[~shelters.index.isin(high_risk_shelters.index)]
med_risk_join = gpd.sjoin(remaining_shelters, buffer_med_gdf, how='left', predicate='within')
med_risk_shelters = med_risk_join[med_risk_join.index_right.notna()].copy()
med_risk_shelters['risk_level'] = 'medium'

# Low risk shelters (not already high or medium risk)
remaining_shelters = shelters[~shelters.index.isin(high_risk_shelters.index) & ~shelters.index.isin(med_risk_shelters.index)]
low_risk_join = gpd.sjoin(remaining_shelters, buffer_low_gdf, how='left', predicate='within')
low_risk_shelters = low_risk_join[low_risk_join.index_right.notna()].copy()
low_risk_shelters['risk_level'] = 'low'

# Safe shelters (not in any buffer)
safe_shelters = shelters[
    ~shelters.index.isin(high_risk_shelters.index) & 
    ~shelters.index.isin(med_risk_shelters.index) & 
    ~shelters.index.isin(low_risk_shelters.index)
].copy()
safe_shelters['risk_level'] = 'safe'

print(f"Risk level assignment completed:")
print(f"- High risk (500m): {len(high_risk_shelters)} shelters")
print(f"- Medium risk (1km): {len(med_risk_shelters)} shelters")
print(f"- Low risk (2km): {len(low_risk_shelters)} shelters")
print(f"- Safe: {len(safe_shelters)} shelters")
print(f"- Total: {len(high_risk_shelters) + len(med_risk_shelters) + len(low_risk_shelters) + len(safe_shelters)} shelters")

In [ ]:
# Combine all shelters with risk levels
print("Combining all shelters with risk levels...")

all_shelters_risk = pd.concat([
    high_risk_shelters[['序號', '縣市及鄉鎮市區', '避難收容處所名稱', '預計收容人數', 'geometry', 'risk_level']],
    med_risk_shelters[['序號', '縣市及鄉鎮市區', '避難收容處所名稱', '預計收容人數', 'geometry', 'risk_level']],
    low_risk_shelters[['序號', '縣市及鄉鎮市區', '避難收容處所名稱', '預計收容人數', 'geometry', 'risk_level']],
    safe_shelters[['序號', '縣市及鄉鎮市區', '避難收容處所名稱', '預計收容人數', 'geometry', 'risk_level']]
])

# Convert back to GeoDataFrame
all_shelters_risk = gpd.GeoDataFrame(all_shelters_risk, crs='EPSG:3826')

print(f"Combined dataset: {len(all_shelters_risk)} shelters")
print(f"\nRisk level distribution:")
print(all_shelters_risk['risk_level'].value_counts())

all_shelters_risk.head()

## 4. Capacity Gap Analysis by Township

**Captain's Log**: Analyzing shelter capacity and risk levels by administrative districts to identify areas with insufficient safe shelter capacity.

In [ ]:
# Spatial join shelters with townships
print("Joining shelters with township boundaries...")

shelters_with_township = gpd.sjoin(all_shelters_risk, townships, how='left', predicate='within')

# Handle shelters that don't match any township
unmatched = shelters_with_township[shelters_with_township.index_right.isna()]
if len(unmatched) > 0:
    print(f"Warning: {len(unmatched)} shelters didn't match any township")
    shelters_with_township = shelters_with_township[shelters_with_township.index_right.notna()]

print(f"Successfully matched {len(shelters_with_township)} shelters to townships")
shelters_with_township.head()

In [ ]:
# Clean capacity data and create township-level statistics
print("Creating township-level risk and capacity analysis...")

# Convert capacity to numeric, handling non-numeric values
shelters_with_township['預計收容人數'] = pd.to_numeric(shelters_with_township['預計收容人數'], errors='coerce').fillna(0)

# Group by township and risk level
township_stats = shelters_with_township.groupby(['T_NAME', 'risk_level']).agg({
    '序號': 'count',  # shelter count
    '預計收容人數': 'sum'  # total capacity
}).rename(columns={'序號': 'shelter_count', '預計收容人數': 'total_capacity'})

# Pivot to get risk levels as columns
township_pivot = township_stats.unstack(fill_value=0)

# Flatten column names
township_pivot.columns = [f'{col[0]}_{col[1]}' for col in township_pivot.columns]

# Calculate risk metrics
township_pivot['total_shelters'] = township_pivot[[f'shelter_count_{risk}' for risk in ['high', 'medium', 'low', 'safe']]].sum(axis=1)
township_pivot['total_capacity'] = township_pivot[[f'total_capacity_{risk}' for risk in ['high', 'medium', 'low', 'safe']]].sum(axis=1)
township_pivot['risk_shelters'] = township_pivot[[f'shelter_count_{risk}' for risk in ['high', 'medium', 'low']]].sum(axis=1)
township_pivot['risk_capacity'] = township_pivot[[f'total_capacity_{risk}' for risk in ['high', 'medium', 'low']]].sum(axis=1)
township_pivot['safe_capacity'] = township_pivot['total_capacity_safe']

# Calculate capacity gap (assuming 30% of population needs evacuation)
# Note: This is a simplified assumption - real analysis would use population data
township_pivot['capacity_gap'] = township_pivot['risk_capacity'] - township_pivot['safe_capacity']
township_pivot['risk_ratio'] = township_pivot['risk_capacity'] / (township_pivot['total_capacity'] + 1)  # +1 to avoid division by zero

print(f"Township analysis completed for {len(township_pivot)} townships")
township_pivot.head()

In [ ]:
# Identify top 10 most at-risk townships
print("Identifying top 10 most at-risk townships...")

# Sort by multiple risk factors
top_risk_townships = township_pivot.sort_values([
    'shelter_count_high',  # prioritize high-risk shelter count
    'risk_capacity',       # then total at-risk capacity
    'capacity_gap'         # then capacity gap
], ascending=False).head(10)

print("\nTop 10 Most At-Risk Townships:")
display_cols = ['shelter_count_high', 'shelter_count_medium', 'shelter_count_low', 'shelter_count_safe',
                'risk_capacity', 'safe_capacity', 'capacity_gap', 'risk_ratio']

print(top_risk_townships[display_cols].round(2))

# Create summary statistics
print(f"\nRisk Summary:")
print(f"Total shelters at risk: {township_pivot['risk_shelters'].sum():,}")
print(f"Total capacity at risk: {township_pivot['risk_capacity'].sum():,}")
print(f"Total safe capacity: {township_pivot['safe_capacity'].sum():,}")
print(f"Overall capacity gap: {township_pivot['capacity_gap'].sum():,}")

## 5. Interactive Risk Map

**Captain's Log**: Creating an interactive map showing river buffers, shelter locations, and risk levels. This will help visualize the spatial distribution of flood risk.

In [ ]:
# Create interactive risk map
print("Creating interactive risk map...")

# Convert data back to WGS84 for folium
rivers_wgs84 = rivers.to_crs('EPSG:4326')
shelters_wgs84 = all_shelters_risk.to_crs('EPSG:4326')
buffer_high_wgs84 = buffer_high_gdf.to_crs('EPSG:4326')
buffer_med_wgs84 = buffer_med_gdf.to_crs('EPSG:4326')
buffer_low_wgs84 = buffer_low_gdf.to_crs('EPSG:4326')
townships_wgs84 = townships.to_crs('EPSG:4326')

# Calculate center of Taiwan for map
center_lat = 23.8
center_lon = 120.9

# Create base map
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=8,
    tiles='OpenStreetMap'
)

# Add township boundaries
folium.GeoJson(
    townships_wgs84,
    style_function=lambda x: {
        'fillColor': 'transparent',
        'color': 'gray',
        'weight': 1,
        'fillOpacity': 0.1
    },
    tooltip=folium.GeoJsonTooltip(fields=['T_NAME'], aliases=['Township'])
).add_to(m)

# Add buffer zones
folium.GeoJson(
    buffer_high_wgs84,
    style_function=lambda x: {
        'fillColor': 'red',
        'color': 'darkred',
        'weight': 2,
        'fillOpacity': 0.3
    },
    tooltip='High Risk Zone (500m)'
).add_to(m)

folium.GeoJson(
    buffer_med_wgs84,
    style_function=lambda x: {
        'fillColor': 'orange',
        'color': 'darkorange',
        'weight': 2,
        'fillOpacity': 0.2
    },
    tooltip='Medium Risk Zone (1km)'
).add_to(m)

folium.GeoJson(
    buffer_low_wgs84,
    style_function=lambda x: {
        'fillColor': 'yellow',
        'color': 'gold',
        'weight': 2,
        'fillOpacity': 0.15
    },
    tooltip='Low Risk Zone (2km)'
).add_to(m)

# Add river polygons
folium.GeoJson(
    rivers_wgs84,
    style_function=lambda x: {
        'fillColor': 'lightblue',
        'color': 'blue',
        'weight': 2,
        'fillOpacity': 0.6
    },
    tooltip='River'
).add_to(m)

print("Base map and layers added successfully")

In [ ]:
# Add shelter points with risk-based coloring
print("Adding shelter points to map...")

# Define colors for risk levels
risk_colors = {
    'high': 'red',
    'medium': 'orange',
    'low': 'gold',
    'safe': 'green'
}

# Add shelters for each risk level
for risk_level, color in risk_colors.items():
    risk_shelters = shelters_wgs84[shelters_wgs84['risk_level'] == risk_level]
    
    for idx, shelter in risk_shelters.iterrows():
        folium.CircleMarker(
            location=[shelter.geometry.y, shelter.geometry.x],
            radius=6,
            popup=f"""
            <b>{shelter['避難收容處所名稱']}</b><br>
            Risk Level: {risk_level.upper()}<br>
            Capacity: {int(shelter['預計收容人數'])} people<br>
            Location: {shelter['縣市及鄉鎮市區']}
            """,
            tooltip=f"{shelter['避難收容處所名稱']} ({risk_level})",
            color=color,
            fillColor=color,
            fillOpacity=0.7,
            weight=2
        ).add_to(m)

print(f"Added {len(shelters_wgs84)} shelter points to map")

# Add legend
legend_html = '''
<div style="position: fixed; 
            bottom: 50px; left: 50px; width: 150px; height: 120px; 
            background-color: white; border:2px solid grey; z-index:9999; 
            font-size:14px; padding: 10px">
<h4>Risk Levels</h4>
<p><i class="fa fa-circle" style="color:red"></i> High (500m)</p>
<p><i class="fa fa-circle" style="color:orange"></i> Medium (1km)</p>
<p><i class="fa fa-circle" style="color:gold"></i> Low (2km)</p>
<p><i class="fa fa-circle" style="color:green"></i> Safe</p>
</div>
'''

m.get_root().add_child(folium.Element(legend_html))

print("Interactive risk map completed!")
display(m)

## 6. Statistical Visualization

**Captain's Log**: Creating static visualizations to complement the interactive map and highlight key findings from the risk analysis.

In [ ]:
# Create bar chart of top 10 at-risk townships
print("Creating Top 10 risk townships visualization...")

plt.figure(figsize=(14, 8))

# Prepare data for plotting
top_10_data = top_risk_townships.reset_index()
x_pos = np.arange(len(top_10_data))

# Create grouped bar chart
width = 0.2

plt.bar(x_pos - width*1.5, top_10_data['shelter_count_high'], width, label='High Risk', color='red', alpha=0.7)
plt.bar(x_pos - width*0.5, top_10_data['shelter_count_medium'], width, label='Medium Risk', color='orange', alpha=0.7)
plt.bar(x_pos + width*0.5, top_10_data['shelter_count_low'], width, label='Low Risk', color='gold', alpha=0.7)
plt.bar(x_pos + width*1.5, top_10_data['shelter_count_safe'], width, label='Safe', color='green', alpha=0.7)

plt.xlabel('Township', fontsize=12)
plt.ylabel('Number of Shelters', fontsize=12)
plt.title('Top 10 Most At-Risk Townships: Shelter Distribution by Risk Level', fontsize=14, fontweight='bold')
plt.xticks(x_pos, top_10_data['T_NAME'], rotation=45, ha='right')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()

# Save the plot
plt.savefig('risk_map.png', dpi=300, bbox_inches='tight')
plt.show()

print("Bar chart saved as 'risk_map.png'")

In [ ]:
# Create capacity analysis chart
print("Creating capacity gap analysis...")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Risk vs Safe Capacity
top_10_sorted = top_10_data.sort_values('risk_capacity', ascending=True)
y_pos = np.arange(len(top_10_sorted))

ax1.barh(y_pos - 0.2, top_10_sorted['risk_capacity'], 0.4, label='At-Risk Capacity', color='coral', alpha=0.7)
ax1.barh(y_pos + 0.2, top_10_sorted['safe_capacity'], 0.4, label='Safe Capacity', color='lightgreen', alpha=0.7)

ax1.set_yticks(y_pos)
ax1.set_yticklabels(top_10_sorted['T_NAME'])
ax1.set_xlabel('Shelter Capacity (people)', fontsize=12)
ax1.set_title('Risk vs Safe Shelter Capacity', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(axis='x', alpha=0.3)

# Plot 2: Capacity Gap
colors = ['red' if gap > 0 else 'green' for gap in top_10_sorted['capacity_gap']]
ax2.barh(top_10_sorted['T_NAME'], top_10_sorted['capacity_gap'], color=colors, alpha=0.7)
ax2.axvline(x=0, color='black', linestyle='-', alpha=0.5)
ax2.set_xlabel('Capacity Gap (At-Risk - Safe)', fontsize=12)
ax2.set_title('Shelter Capacity Gap Analysis', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('capacity_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Capacity analysis charts saved as 'capacity_analysis.png'")

## 7. Export Results

**Captain's Log**: Exporting the shelter risk audit data and finalizing the analysis. Creating JSON output for integration with other systems.

In [ ]:
# Export shelter risk audit data
print("Exporting shelter risk audit data...")

# Prepare shelter data for export
shelter_audit = all_shelters_risk[['序號', '縣市及鄉鎮市區', '避難收容處所名稱', '預計收容人數', 'risk_level']].copy()
shelter_audit.columns = ['shelter_id', 'location', 'name', 'capacity', 'risk_level']

# Convert to WGS84 coordinates for export
shelters_wgs84_for_export = all_shelters_risk.to_crs('EPSG:4326')
shelter_audit['longitude'] = shelters_wgs84_for_export.geometry.x
shelter_audit['latitude'] = shelters_wgs84_for_export.geometry.y

# Convert to JSON-serializable format
shelter_audit_json = shelter_audit.to_dict('records')

# Add metadata
audit_metadata = {
    "analysis_date": pd.Timestamp.now().isoformat(),
    "buffer_parameters": {
        "high_risk_m": BUFFER_HIGH,
        "medium_risk_m": BUFFER_MED,
        "low_risk_m": BUFFER_LOW
    },
    "summary": {
        "total_shelters": len(all_shelters_risk),
        "high_risk_shelters": len(all_shelters_risk[all_shelters_risk['risk_level'] == 'high']),
        "medium_risk_shelters": len(all_shelters_risk[all_shelters_risk['risk_level'] == 'medium']),
        "low_risk_shelters": len(all_shelters_risk[all_shelters_risk['risk_level'] == 'low']),
        "safe_shelters": len(all_shelters_risk[all_shelters_risk['risk_level'] == 'safe']),
        "total_at_risk_capacity": int(all_shelters_risk[all_shelters_risk['risk_level'].isin(['high', 'medium', 'low'])]['預計收容人數'].sum()),
        "total_safe_capacity": int(all_shelters_risk[all_shelters_risk['risk_level'] == 'safe']['預計收容人數'].sum())
    }
}

# Combine metadata and shelter data
final_output = {
    "metadata": audit_metadata,
    "shelters": shelter_audit_json
}

# Save to JSON file
with open('shelter_risk_audit.json', 'w', encoding='utf-8') as f:
    json.dump(final_output, f, ensure_ascii=False, indent=2)

print(f"Shelter risk audit exported to 'shelter_risk_audit.json'")
print(f"Exported {len(shelter_audit_json)} shelter records")

# Display sample of exported data
print("\nSample of exported data:")
print(json.dumps(final_output['metadata'], indent=2, ensure_ascii=False))
print("\nFirst shelter record:")
print(json.dumps(shelter_audit_json[0], indent=2, ensure_ascii=False))

## 8. Summary and Key Findings

**Captain's Log**: Completing the ARIA analysis. The system has successfully identified flood risks and capacity gaps across Taiwan's shelter network.

In [ ]:
# Generate final summary
print("=" * 60)
print("AUTOMATED REGIONAL IMPACT AUDITOR (ARIA) - FINAL REPORT")
print("=" * 60)

print(f"\n📊 ANALYSIS SUMMARY:")
print(f"• Total shelters analyzed: {len(all_shelters_risk):,}")
print(f"• High risk (500m): {len(all_shelters_risk[all_shelters_risk['risk_level'] == 'high']):,} shelters")
print(f"• Medium risk (1km): {len(all_shelters_risk[all_shelters_risk['risk_level'] == 'medium']):,} shelters")
print(f"• Low risk (2km): {len(all_shelters_risk[all_shelters_risk['risk_level'] == 'low']):,} shelters")
print(f"• Safe zones: {len(all_shelters_risk[all_shelters_risk['risk_level'] == 'safe']):,} shelters")

print(f"\n🚨 CAPACITY ANALYSIS:")
total_risk_capacity = all_shelters_risk[all_shelters_risk['risk_level'].isin(['high', 'medium', 'low'])]['預計收容人數'].sum()
total_safe_capacity = all_shelters_risk[all_shelters_risk['risk_level'] == 'safe']['預計收容人數'].sum()
capacity_gap = total_risk_capacity - total_safe_capacity

print(f"• Total at-risk capacity: {int(total_risk_capacity):,} people")
print(f"• Total safe capacity: {int(total_safe_capacity):,} people")
print(f"• Capacity gap: {int(capacity_gap):,} people")

if capacity_gap > 0:
    print(f"• ⚠️  WARNING: Insufficient safe capacity by {int(capacity_gap):,} people!")
else:
    print(f"• ✅ Sufficient safe capacity available")

print(f"\n🏆 TOP 5 HIGHEST RISK TOWNSHIPS:")
for i, (idx, row) in enumerate(top_risk_townships.head(5).iterrows(), 1):
    print(f"{i}. {idx}: {int(row['shelter_count_high'])} high-risk, {int(row['risk_capacity']):,} at-risk capacity")

print(f"\n📁 OUTPUT FILES GENERATED:")
print(f"• shelter_risk_audit.json - Complete shelter risk database")
print(f"• risk_map.png - Top 10 risk townships visualization")
print(f"• capacity_analysis.png - Capacity gap analysis")
print(f"• Interactive map displayed above")

print(f"\n🎯 RECOMMENDATIONS:")
if capacity_gap > 0:
    print(f"• 1. Prioritize capacity expansion in top 10 high-risk townships")
    print(f"• 2. Establish additional safe zones away from river buffers")
    print(f"• 3. Develop evacuation plans for {int(capacity_gap):,} at-risk persons")
else:
    print(f"• 1. Maintain current shelter network capacity")
    print(f"• 2. Focus on preparedness in high-risk zones")
    print(f"• 3. Regular monitoring of river flood risks")

print(f"\n" + "=" * 60)
print(f"ARIA ANALYSIS COMPLETE - {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)